# Project 29 — Citation Verifier for RAG

## Check if LLM Answers Are Supported by Retrieved Chunks

**Goal:** Extract claims from LLM-generated answers, verify each against
source evidence, and produce a groundedness report.

**Stack:** Ollama · LangChain · Jupyter

### What You'll Learn
1. Claim extraction from generated answers
2. Evidence-based verification of individual claims
3. Groundedness scoring (supported / contradicted / not mentioned)
4. Auto-revision of unsupported claims

### Prerequisites
- **Ollama** with `qwen3:8b`

In [ ]:
!pip install -q langchain langchain-ollama

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests
OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    print(f"Ollama running — {len(r.json().get('models',[]))} model(s)")
except Exception as e:
    print(f"Cannot reach Ollama: {e}")

## Step 2 — Test Cases

Answer–source pairs with known verdicts: fully supported,
partially supported, and not supported.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.0)

cases = [
    {"q": "Capital of France?",
     "answer": "The capital of France is Paris. It has about 2.1 million people.",
     "sources": ["Paris is the capital and most populous city of France, pop 2,102,650."],
     "expected": "supported"},
    {"q": "When was Python created?",
     "answer": "Python was created in 1991 by Guido van Rossum. It was designed at Microsoft.",
     "sources": ["Python was conceived in the late 1980s by Guido van Rossum at CWI Netherlands."],
     "expected": "partial"},
    {"q": "GPT-4 speed?",
     "answer": "GPT-4 processes 1 million tokens per second and costs $0.001 per query.",
     "sources": ["GPT-4 is a large multimodal model created by OpenAI, released March 2023."],
     "expected": "not_supported"},
]
print(f"{len(cases)} test cases")

## Step 3 — Claim Extraction

In [ ]:
cp = ChatPromptTemplate.from_messages([
    ("system", "Extract individual factual claims from the answer. "
     "One claim per line, no numbering."),
    ("human", "Answer: {answer}")
])
cc = cp | llm | StrOutputParser()

for tc in cases:
    r = cc.invoke({"answer": tc["answer"]})
    tc["claims"] = [c.strip() for c in r.strip().split("\n") if c.strip()]
    print(f"Q: {tc['q']}")
    print(f"  Claims ({len(tc['claims'])}):'")
    for c in tc["claims"]: print(f"    • {c}")
    print()

## Step 4 — Verify Each Claim Against Sources

In [ ]:
vp = ChatPromptTemplate.from_messages([
    ("system", """Fact-check this claim against the source evidence.
Return:
VERDICT: supported / contradicted / not_mentioned
EVIDENCE: relevant quote or 'none'"""),
    ("human", "Claim: {claim}\n\nSource:\n{sources}")
])
vc = vp | llm | StrOutputParser()

for tc in cases:
    src = "\n".join(tc["sources"])
    print(f"\nQ: {tc['q']} (expected: {tc['expected']})")
    verdicts = []
    for claim in tc["claims"]:
        r = vc.invoke({"claim": claim, "sources": src})
        v = "not_mentioned"
        for line in r.split("\n"):
            if "VERDICT:" in line.upper():
                vl = line.split(":", 1)[1].strip().lower()
                if "support" in vl: v = "supported"
                elif "contradict" in vl: v = "contradicted"
        verdicts.append(v)
        icon = {"supported": "✓", "contradicted": "✗", "not_mentioned": "?"}
        print(f"  {icon.get(v, '?')} [{v}] {claim}")
    sup = sum(1 for v in verdicts if v == "supported")
    tc["groundedness"] = sup / max(len(verdicts), 1)
    print(f"  Groundedness: {tc['groundedness']:.0%}")

## Step 5 — Dashboard

In [ ]:
print("="*60)
print("CITATION VERIFICATION DASHBOARD")
print("="*60)
print(f"\n{'Question':<35} {'Expected':<18} {'Score':>8}")
print("-" * 63)
for tc in cases:
    print(f"  {tc['q'][:33]:<33} {tc['expected']:<18} {tc['groundedness']:>7.0%}")
avg = sum(tc["groundedness"] for tc in cases) / len(cases)
print(f"\nAverage groundedness: {avg:.0%}")

## Step 6 — Auto-Revise Unsupported Claims

In [ ]:
rp = ChatPromptTemplate.from_messages([
    ("system", "Rewrite using ONLY information from the sources. "
     "Remove any claims not supported by evidence."),
    ("human", "Original: {answer}\n\nSources:\n{sources}\n\nRevised:")
])
rc = rp | llm | StrOutputParser()

tc = cases[1]  # partially supported
revised = rc.invoke({"answer": tc["answer"], "sources": "\n".join(tc["sources"])})
print(f"Q: {tc['q']}")
print(f"Original: {tc['answer']}")
print(f"Revised:  {revised}")

## Limitations

1. **LLM-as-judge bias:** The verifier LLM may make errors on nuanced claims.
2. **Claim granularity:** Over-splitting creates trivial claims.
3. **Source quality assumed:** We assume sources are correct ground truth.

## What You Learned

| Concept | What We Did |
|---|---|
| **Claim extraction** | Atomic verifiable statements |
| **Evidence verification** | Check each claim vs source |
| **Groundedness scoring** | Fraction of supported claims |
| **Auto-revision** | Rewrite to remove unsupported claims |